# 02 — Data Cleaning
**QM640 Data Analytics Capstone | Walsh College**

This notebook cleans the raw Fragrantica dataset. Steps:
1. Remove exact duplicates
2. Remove near-duplicates using composite key (perfume_name + brand)
3. Handle missing values
4. Filter outliers in rating_score and num_votes
5. Standardise text fields
6. Save cleaned dataset to `data/processed/fragrantica_clean.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

RAW_PATH       = Path("../data/raw/fragrantica_raw.csv")
PROCESSED_PATH = Path("../data/processed/fragrantica_clean.csv")
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW_PATH)
print(f"Raw shape: {df.shape}")

## 2.1 Remove Duplicates

In [ ]:
before = len(df)
df = df.drop_duplicates()                                      # exact duplicates
df = df.drop_duplicates(subset=["perfume_name", "brand"])     # near-duplicates
print(f"Removed {before - len(df)} duplicates. Remaining: {len(df):,}")

## 2.2 Drop Records with No Rating

In [ ]:
before = len(df)
df = df.dropna(subset=["rating_score"])
print(f"Dropped {before - len(df)} rows with missing rating_score. Remaining: {len(df):,}")

## 2.3 Filter Low-Vote Records (< 5 votes)

In [ ]:
before = len(df)
df = df[df["num_votes"] >= 5]
print(f"Dropped {before - len(df)} low-vote records. Remaining: {len(df):,}")

## 2.4 Standardise Text Fields

In [ ]:
str_cols = ["perfume_name", "brand", "concentration", "gender_label"]
for col in str_cols:
    df[col] = df[col].str.strip().str.title()

# Standardise gender labels
gender_map = {
    "For Women": "Women",
    "For Men": "Men",
    "Unisex": "Unisex",
    "For Women And Men": "Unisex",
}
df["gender_label"] = df["gender_label"].map(gender_map).fillna("Unknown")

print(df["gender_label"].value_counts())

## 2.5 Check Release Year Range

In [ ]:
df["release_year"] = pd.to_numeric(df["release_year"], errors="coerce")
df = df[(df["release_year"] >= 1950) | df["release_year"].isna()]
print(df["release_year"].describe())

## 2.6 Missing Value Summary

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
print(missing[missing > 0])

## 2.7 Save Cleaned Dataset

In [ ]:
df.to_csv(PROCESSED_PATH, index=False)
print(f"Cleaned dataset saved to {PROCESSED_PATH}")
print(f"Final shape: {df.shape}")